In [1]:
import oceanbench

oceanbench.__version__

'0.5.1'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# glowcascade_final over the full official start set, cut to NINE lead days.
#
# 52 Wednesday challenger folders of 2024, 20240103 through 20241225, each
# initialised from the as-issued GLO12 nowcast of the Tuesday before it and
# forced by the IFS forecast issued that same Tuesday.
#
# WHY NINE. The IFS forecast package carries lead_day_index 0..9, that is the
# forcing of forecast days 1..10 minus its last day, so the tenth forecast day
# is driven by PERSISTED lead 9 forcing rather than by a forecast. Julien's
# decision of 2026-09-10: the entry scores the nine days that are forced by a
# real IFS forecast and stops there. Nothing is recomputed and no forecast
# zarr is touched: the store still holds ten days per start, this module drops
# the last time step on the way in.
#
# This copy is scored under oceanbench 0.5.1.
import datetime
import pathlib

import xarray

_ROOT = pathlib.Path("/mnt/data/glonet2/ifs21/forecasts/glowcascade_v4")
_PATHS = sorted(_ROOT.glob("2024*.zarr"))
_FIRST_DAYS = [datetime.datetime.strptime(p.stem, "%Y%m%d") for p in _PATHS]
_LEAD_DAYS = 9


def _prepared(dataset: xarray.Dataset) -> xarray.Dataset:
    dataset = dataset.isel(time=slice(0, _LEAD_DAYS))
    lead_count = dataset.sizes["time"]
    return dataset.rename({"time": "lead_day_index"}).assign_coords({"lead_day_index": range(lead_count)})


challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [str(p) for p in _PATHS],
    engine="zarr",
    preprocess=_prepared,
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=False,
).assign_coords({"first_day_datetime": _FIRST_DAYS})


### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.066685,0.067074,0.067105,0.067191,0.067572,0.068218,0.069165,0.070448,0.071487
Temperature (°C) [sea_water_potential_temperature]{surface},0.516949,0.517569,0.518079,0.520400,0.524943,0.532740,0.543826,0.556202,0.566310
Salinity (PSU) [sea_water_salinity]{surface},0.579295,0.575311,0.571345,0.567893,0.564352,0.561378,0.558978,0.556788,0.553978
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.118492,0.119157,0.120001,0.121241,0.122870,0.125188,0.128118,0.131439,0.134115
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.119525,0.120206,0.121153,0.122541,0.124637,0.127319,0.130472,0.133799,0.136431
Temperature (°C) [sea_water_potential_temperature]{50m},0.864315,0.864207,0.863156,0.862138,0.862769,0.864749,0.868286,0.873151,0.875664
Salinity (PSU) [sea_water_salinity]{50m},0.242672,0.242555,0.242241,0.241924,0.241776,0.241819,0.242015,0.242260,0.242206
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.112320,0.112874,0.113264,0.113783,0.114559,0.115758,0.117397,0.119308,0.120592
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.113279,0.113460,0.113686,0.114108,0.114762,0.115791,0.117345,0.119123,0.120312
Temperature (°C) [sea_water_potential_temperature]{100m},1.058971,1.059690,1.059548,1.059887,1.061177,1.064326,1.069632,1.074809,1.077028


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},41.771454,42.116581,42.402897,42.65175,42.954651,43.295983,43.690118,44.092566,44.298396


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.116709,0.117297,0.117454,0.117957,0.118809,0.119743,0.121767,0.123678,0.124975
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.122656,0.123091,0.123460,0.123344,0.124126,0.124787,0.127020,0.129351,0.131113


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Temperature (°C) [sea_water_potential_temperature]{surface},0.781312,0.809498,0.779326,0.795823,0.823603,0.864932,0.856872,0.873423,0.901574
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.733275,0.745434,0.765108,0.786667,0.787046,0.801708,0.818479,0.813116,0.808008
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.863667,0.876760,0.860851,0.892588,0.882990,0.889929,0.910194,0.936261,0.936553
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.779565,0.802858,0.791824,0.795119,0.813680,0.811730,0.834637,0.832545,0.863819
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.518690,0.533641,0.528491,0.529647,0.542248,0.557462,0.557496,0.567100,0.586158
Salinity (PSU) [sea_water_salinity]{0-5m},0.257618,0.276999,0.272249,0.301253,0.274338,0.285833,0.269526,0.269246,0.290781
Salinity (PSU) [sea_water_salinity]{5-100m},0.271607,0.267417,0.278684,0.262613,0.300852,0.287243,0.274224,0.288029,0.282987
Salinity (PSU) [sea_water_salinity]{100-300m},0.127376,0.129172,0.130250,0.130261,0.133667,0.132829,0.137515,0.133991,0.136907
Salinity (PSU) [sea_water_salinity]{300-600m},0.081304,0.081932,0.080856,0.081695,0.082676,0.084102,0.084451,0.087217,0.088555
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.048448,0.049417,0.050151,0.051622,0.052846,0.054643,0.056356,0.058152,0.060422


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},9.968276,19.270359,28.1826,36.806114,45.21637,53.449123,61.530106


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.013604,0.016525,0.018998,0.022179,0.026049,0.030109,0.034416,0.038787,0.041943
Temperature (°C) [sea_water_potential_temperature]{surface},0.171973,0.198628,0.226548,0.255792,0.286936,0.319743,0.355214,0.389822,0.416067
Salinity (PSU) [sea_water_salinity]{surface},0.127490,0.146356,0.162066,0.176211,0.189487,0.204580,0.219815,0.233891,0.245281
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.048678,0.054970,0.061621,0.069174,0.077339,0.085726,0.094669,0.103118,0.109438
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.050307,0.057041,0.063963,0.071415,0.079731,0.088452,0.097276,0.105798,0.112233
Temperature (°C) [sea_water_potential_temperature]{50m},0.303754,0.329021,0.354686,0.383143,0.415352,0.451622,0.490258,0.525832,0.548077
Salinity (PSU) [sea_water_salinity]{50m},0.062894,0.067535,0.072860,0.078778,0.085415,0.092668,0.100267,0.107405,0.112258
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.043710,0.047890,0.052843,0.058356,0.064703,0.071617,0.079076,0.086150,0.091108
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.045753,0.049875,0.054685,0.060169,0.066307,0.073039,0.080226,0.087257,0.092350
Temperature (°C) [sea_water_potential_temperature]{100m},0.272837,0.304725,0.338558,0.377315,0.419513,0.464845,0.512227,0.555516,0.582698


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},31.937247,32.901371,33.672301,34.471557,35.301269,36.173129,37.069545,37.851666,38.494574


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.043921,0.050680,0.057678,0.064943,0.072103,0.079556,0.087184,0.094222,0.099100
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.045668,0.053447,0.061576,0.069534,0.077245,0.085319,0.093273,0.100424,0.105182


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},3.995432,7.912033,12.019097,16.48181,21.367662,26.691301,32.46434
